In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/soheiltehranipour/snappfood-persian-sentiment-analysis/Snappfood - Sentiment Analysis.csv
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__results__.html
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__huggingface_repos__.json
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__output__.json
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/custom.css
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm/config.json
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm/training_args.bin
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm/tokenizer.json
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm/tokenizer_config.json
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm/model.safetensors
/kaggle/inpu

In [13]:
# ============================================================
# Step 1: Setup, Seeds, Paths, Tokenizer
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)
from datasets import Dataset, DatasetDict

# ------------------------------
# Full Reproducibility
# ------------------------------
SEED = 42

def set_full_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)          # transformers
    os.environ["PYTHONHASHSEED"] = str(seed)

set_full_seed(SEED)

# ------------------------------
# Device & Basic Info
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------
# Paths (exact as provided)
# ------------------------------
DATA_PATH = "/kaggle/input/datasets/soheiltehranipour/snappfood-persian-sentiment-analysis/Snappfood - Sentiment Analysis.csv"

BASE_MODEL_NAME = "sbunlp/fabert"
KG_MODEL_PATH   = "/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm"

# Output directories
OUTPUT_BASE = "/kaggle/working/fabert_base_sentiment"
OUTPUT_KG   = "/kaggle/working/fabert_kg_sentiment"

os.makedirs(OUTPUT_BASE, exist_ok=True)
os.makedirs(OUTPUT_KG, exist_ok=True)

print("Paths set successfully.")

# ------------------------------
# Load Tokenizer (shared for both models)
# ------------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
print(f"Tokenizer loaded from: {BASE_MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model max length (config): {tokenizer.model_max_length}")

# Quick test
sample = "کیفیت غذا عالی بود ممنون"
print("Tokenize test:", tokenizer.tokenize(sample)[:10])


Device: cpu
Paths set successfully.
Tokenizer loaded from: sbunlp/fabert
Vocab size: 50000
Model max length (config): 512
Tokenize test: ['کیفیت', 'غذا', 'عالی', 'بود', 'ممنون']


In [14]:
# ============================================================
# Diagnostic: Inspect the raw CSV file
# ============================================================

print("File exists:", os.path.exists(DATA_PATH))
print("File size (MB):", round(os.path.getsize(DATA_PATH) / 1024**2, 2))

print("\n----- First 10 lines of raw file -----")
with open(DATA_PATH, "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(f"Line {i}: {repr(line[:200])}")   # show raw content


File exists: True
File size (MB): 11.35

----- First 10 lines of raw file -----
Line 0: '\tcomment\tlabel\tlabel_id\n'
Line 1: '\tواقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح\tSAD\t1\n'
Line 2: '\tقرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از موقع رسید، شما ببین چقدرررررررررررر پلاک خفنههههه، من سالهاست مشتریشونم و سالهاست مزه بهشت میده غذاشون\tHAPPY\t0\n'
Line 3: '\tقیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط ظاهر فریبنده داره، پرش میکنن کالباس و قارچ\tSAD\t1\n'
Line 4: '\tعالللی بود همه چه درست و به اندازه و کیفیت خوب، امیداورم همیشه کیفیتتون خوب باشه ما مشتری همیشگی بشیم\tHAPPY\t0\n'
Line 5: '\tشیرینی وانیلی فقط یک مدل بود.\tHAPPY\t0\n'
Line 6: '\tبدترین پیتزایی که تا به حال خورده بودم\tSAD\t1\n'
Line 7: '\tاز همه لحاظ عالی ممنونم\tHAPPY\t0\n'
Line 8: '\tکیفیت غذا متوسط رو به پایین بود انگار داخل یه رستوران معمولی غذا خوردی درحالی که امتیاز رستوران در اسنپ فود ۴٫۳ بود\tSAD\t1\n'
Line 9: '\tهمه اقلام تازه و به روز وخیلیییییی سریع بدستم رسید واقعا متشکرم\tHAPPY\t0\n'


In [15]:
# ============================================================
# Step 2 (Final Fixed): Load + Clean + Stratified Split
# ============================================================

# ------------------------------
# 1. Load CSV correctly (tab-separated + leading tab)
# ------------------------------
df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip"
)

print("Raw shape:", df.shape)
print("Raw columns:", df.columns.tolist())
print("\nFirst 3 rows (raw):")
print(df.head(3))

# ------------------------------
# 2. Clean columns
# ------------------------------
# Because of leading tab, first column is usually "Unnamed: 0" or empty
df.columns = [str(c).strip() for c in df.columns]

# Drop completely empty / unnamed columns
unnamed_cols = [c for c in df.columns if c.startswith("Unnamed") or c == ""]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)
    print(f"Dropped unnamed columns: {unnamed_cols}")

print("Columns after cleanup:", df.columns.tolist())

# Make sure we have the expected columns
expected = {"comment", "label", "label_id"}
if not expected.issubset(set(df.columns)):
    raise ValueError(f"Unexpected columns: {df.columns.tolist()}")

# Keep only what we need
df = df[["comment", "label_id"]].copy()
df = df.rename(columns={"comment": "text", "label_id": "labels"})

# ------------------------------
# 3. Basic cleaning
# ------------------------------
df = df.dropna(subset=["text", "labels"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 3].reset_index(drop=True)

# Convert labels to integer 0/1
df["labels"] = pd.to_numeric(df["labels"], errors="coerce")
df = df.dropna(subset=["labels"])
df["labels"] = df["labels"].astype(int)

# Keep only valid binary labels
df = df[df["labels"].isin([0, 1])].reset_index(drop=True)

print("\nAfter cleaning shape:", df.shape)
print("Label distribution (0=HAPPY, 1=SAD):")
print(df["labels"].value_counts().sort_index())
print(df["labels"].value_counts(normalize=True).sort_index().round(3))

print("\nSample texts:")
print(df.head(3))

# ------------------------------
# 4. Stratified Split (70% / 15% / 15%) - No Data Leakage
# ------------------------------
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["labels"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["labels"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("\n" + "="*55)
print("Split sizes:")
print(f"Train : {len(train_df):,} ({len(train_df)/len(df):.1%})")
print(f"Val   : {len(val_df):,} ({len(val_df)/len(df):.1%})")
print(f"Test  : {len(test_df):,} ({len(test_df)/len(df):.1%})")

print("\nLabel distribution per split:")
for name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name}:")
    print(split_df["labels"].value_counts().sort_index())
    print("Ratio:", split_df["labels"].value_counts(normalize=True).sort_index().round(3).to_dict())

# ------------------------------
# 5. Convert to Hugging Face DatasetDict
# ------------------------------
dataset = DatasetDict({
    "train":      Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test":       Dataset.from_pandas(test_df)
})

# Remove any leftover index columns
cols_to_remove = [c for c in dataset["train"].column_names if "index" in c.lower() or c.startswith("__")]
if cols_to_remove:
    dataset = dataset.remove_columns(cols_to_remove)

print("\n" + "="*55)
print("Final DatasetDict:")
print(dataset)

print("\nOne training example:")
print(dataset["train"][0])

# Save splits for full reproducibility
train_df.to_csv("/kaggle/working/train_split.csv", index=False)
val_df.to_csv("/kaggle/working/val_split.csv", index=False)
test_df.to_csv("/kaggle/working/test_split.csv", index=False)
print("\nSplits saved to /kaggle/working/")


Raw shape: (70000, 4)
Raw columns: ['Unnamed: 0', 'comment', 'label', 'label_id']

First 3 rows (raw):
  Unnamed: 0                                            comment  label  \
0        NaN    واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح    SAD   
1        NaN  قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از مو...  HAPPY   
2        NaN  قیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط...    SAD   

   label_id  
0       1.0  
1       0.0  
2       1.0  
Dropped unnamed columns: ['Unnamed: 0']
Columns after cleanup: ['comment', 'label', 'label_id']

After cleaning shape: (69480, 2)
Label distribution (0=HAPPY, 1=SAD):
labels
0    34916
1    34564
Name: count, dtype: int64
labels
0    0.503
1    0.497
Name: proportion, dtype: float64

Sample texts:
                                                text  labels
0    واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح       1
1  قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از مو...       0
2  قیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط...       1



In [16]:
# ============================================================
# Step 3: Tokenization (shared tokenizer for both models)
# ============================================================

MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,          # dynamic padding later with DataCollator
    )

# Apply tokenization to all splits
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],   # text is no longer needed after tokenization
    desc="Tokenizing"
)

print("Tokenized DatasetDict:")
print(tokenized_dataset)

print("\nSample tokenized example (train[0]):")
print(tokenized_dataset["train"][0])

# Check lengths
print("\n----- Length statistics (train set) -----")
lengths = [len(x) for x in tokenized_dataset["train"]["input_ids"]]
print(f"Min length : {min(lengths)}")
print(f"Max length : {max(lengths)}")
print(f"Mean length: {np.mean(lengths):.1f}")
print(f"Median     : {np.median(lengths):.1f}")

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\nDataCollatorWithPadding is ready.")
print("Tokenization completed successfully.")


Tokenizing:   0%|          | 0/48636 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/10422 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/10422 [00:00<?, ? examples/s]

Tokenized DatasetDict:
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 48636
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10422
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10422
    })
})

Sample tokenized example (train[0]):
{'labels': 0, 'input_ids': [101, 2533, 3196, 2808, 622, 8014, 2303, 35720, 20509, 7294, 7227, 2299, 4449, 6721, 2299, 13355, 2516, 2307, 2297, 9078, 2608, 2434, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

----- Length statistics (train set) -----
Min length : 6
Max length : 128
Mean length: 23.6
Median     : 18.0

DataCollatorWithPadding is ready.
Tokenization completed successfull

In [ ]:
# ============================================================
# FINAL BLOCK: Train Base + Train KG + Test + Compare + Plots
# Assumes Step 1, Step 2, Step 3 are already done
# ============================================================

import os
import gc
import json
import inspect
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

# ------------------------------
# 0) Safety checks
# ------------------------------
required = [
    "SEED", "device",
    "BASE_MODEL_NAME", "KG_MODEL_PATH",
    "OUTPUT_BASE", "OUTPUT_KG",
    "tokenizer", "tokenized_dataset", "data_collator",
]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(
        f"Missing: {missing}\n"
        "Re-run Step 1, Step 2, and Step 3 first."
    )

set_seed(SEED)
os.makedirs(OUTPUT_BASE, exist_ok=True)
os.makedirs(OUTPUT_KG, exist_ok=True)
os.makedirs("/kaggle/working/figures", exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

ID2LABEL = {0: "HAPPY", 1: "SAD"}
LABEL2ID = {"HAPPY": 0, "SAD": 1}
NUM_LABELS = 2

print("Device:", device)
print("Train size:", len(tokenized_dataset["train"]))
print("Val size  :", len(tokenized_dataset["validation"]))
print("Test size :", len(tokenized_dataset["test"]))
print("KG exists :", os.path.exists(KG_MODEL_PATH))


# ------------------------------
# 1) Helpers
# ------------------------------
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_classification_model(model_path_or_name: str, model_tag: str):
    print("\n" + "=" * 60)
    print(f"Loading: {model_tag}")
    print(f"Source : {model_path_or_name}")
    print("=" * 60)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path_or_name,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Model type      : {model.__class__.__name__}")
    print(f"Num labels      : {model.config.num_labels}")
    print(f"Hidden size     : {model.config.hidden_size}")
    print(f"Num layers      : {model.config.num_hidden_layers}")
    print(f"Vocab size      : {model.config.vocab_size}")
    print(f"Total params    : {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    print(f"Device          : {next(model.parameters()).device}")
    return model


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


def build_training_args(output_dir: str) -> TrainingArguments:
    """Version-safe TrainingArguments for Kaggle transformers."""
    candidate_args = {
        "output_dir": output_dir,
        "num_train_epochs": 2,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "linear",
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 32,
        "gradient_accumulation_steps": 1,
        "optim": "adamw_torch",
        "eval_strategy": "epoch",
        "evaluation_strategy": "epoch",
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "save_total_limit": 2,
        "logging_strategy": "steps",
        "logging_steps": 100,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "fp16": torch.cuda.is_available(),
        "dataloader_num_workers": 2,
        "remove_unused_columns": True,
        "overwrite_output_dir": True,
    }

    valid_params = inspect.signature(TrainingArguments.__init__).parameters
    filtered = {k: v for k, v in candidate_args.items() if k in valid_params}

    # Prefer new eval key if both exist
    if "eval_strategy" in filtered and "evaluation_strategy" in filtered:
        filtered.pop("evaluation_strategy", None)

    try:
        return TrainingArguments(**filtered)
    except TypeError:
        # Fallback: drop optional keys that may still break
        for k in ["optim", "overwrite_output_dir", "data_seed", "report_to"]:
            filtered.pop(k, None)
        if "eval_strategy" in filtered and "evaluation_strategy" not in filtered:
            # older API may only accept evaluation_strategy
            if "evaluation_strategy" in valid_params:
                filtered.pop("eval_strategy", None)
                filtered["evaluation_strategy"] = "epoch"
        return TrainingArguments(**filtered)


def extract_history(trainer: Trainer):
    """Extract train/eval curves from trainer state."""
    rows = []
    for item in trainer.state.log_history:
        rows.append(item)
    return pd.DataFrame(rows)


def evaluate_on_split(trainer: Trainer, split_name: str):
    """Return metrics + predictions for a dataset split."""
    pred_out = trainer.predict(tokenized_dataset[split_name])
    logits = pred_out.predictions
    labels = pred_out.label_ids
    preds = np.argmax(logits, axis=-1)

    metrics = {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
        "weighted_f1": float(f1_score(labels, preds, average="weighted")),
    }
    report = classification_report(
        labels, preds,
        target_names=["HAPPY", "SAD"],
        digits=4,
        zero_division=0,
    )
    cm = confusion_matrix(labels, preds)
    return metrics, report, cm, preds, labels


def train_one_model(model_name_or_path: str, model_tag: str, output_dir: str):
    """
    Full fine-tuning pipeline for one model under identical settings.
    Returns trainer, history, val metrics, test metrics, reports, CMs.
    """
    free_memory()
    set_seed(SEED)

    print("\n" + "#" * 70)
    print(f"START TRAINING: {model_tag}")
    print("#" * 70)

    model = load_classification_model(model_name_or_path, model_tag)
    args = build_training_args(output_dir)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    train_result = trainer.train()
    print(f"\n[{model_tag}] train() finished.")
    print(train_result.metrics)

    # Save best model + tokenizer
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    history_df = extract_history(trainer)
    history_path = os.path.join("/kaggle/working/results", f"{model_tag}_history.csv")
    history_df.to_csv(history_path, index=False)

    # Validation / Test evaluation
    val_metrics, val_report, val_cm, _, _ = evaluate_on_split(trainer, "validation")
    test_metrics, test_report, test_cm, test_preds, test_labels = evaluate_on_split(trainer, "test")

    print(f"\n[{model_tag}] Validation metrics:")
    print(val_metrics)
    print(f"\n[{model_tag}] Test metrics:")
    print(test_metrics)
    print(f"\n[{model_tag}] Test classification report:")
    print(test_report)

    # Persist metrics
    result_payload = {
        "model_tag": model_tag,
        "source": model_name_or_path,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "train_runtime_metrics": train_result.metrics,
    }
    with open(os.path.join("/kaggle/working/results", f"{model_tag}_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(result_payload, f, ensure_ascii=False, indent=2)

    # Persist test predictions
    pred_df = pd.DataFrame({
        "gold_label": test_labels,
        "pred_label": test_preds,
        "gold_name": [ID2LABEL[int(x)] for x in test_labels],
        "pred_name": [ID2LABEL[int(x)] for x in test_preds],
    })
    pred_df.to_csv(
        os.path.join("/kaggle/working/results", f"{model_tag}_test_predictions.csv"),
        index=False
    )

    # Free model from trainer to reduce memory before next run
    out = {
        "model_tag": model_tag,
        "trainer": trainer,
        "history_df": history_df,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "val_report": val_report,
        "test_report": test_report,
        "val_cm": val_cm,
        "test_cm": test_cm,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "output_dir": output_dir,
    }

    # Keep only necessary objects; delete heavy model ref explicitly later by caller
    return out


def plot_confusion(cm, title, save_path):
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["HAPPY", "SAD"],
        yticklabels=["HAPPY", "SAD"],
    )
    plt.xlabel("Predicted")
    plt.ylabel("Gold")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()


def plot_metric_bars(df_compare, save_path):
    metrics = ["accuracy", "macro_f1", "weighted_f1"]
    x = np.arange(len(metrics))
    width = 0.35

    base_vals = [df_compare.loc["FaBERT-Base", m] for m in metrics]
    kg_vals = [df_compare.loc["FaBERT-KG", m] for m in metrics]

    plt.figure(figsize=(8, 5))
    b1 = plt.bar(x - width / 2, base_vals, width, label="FaBERT-Base")
    b2 = plt.bar(x + width / 2, kg_vals, width, label="FaBERT-KG")

    plt.xticks(x, ["Accuracy", "Macro-F1", "Weighted-F1"])
    plt.ylim(0.0, 1.0)
    plt.ylabel("Score")
    plt.title("Test Set Comparison")
    plt.legend()

    for bar in list(b1) + list(b2):
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}",
                 ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()


def plot_loss_curves(history_df, model_tag, save_path):
    plt.figure(figsize=(8, 5))

    if "loss" in history_df.columns:
        train_loss = history_df.dropna(subset=["loss"])
        if len(train_loss) > 0:
            plt.plot(train_loss["step"], train_loss["loss"], label="train_loss")

    if "eval_loss" in history_df.columns:
        eval_loss = history_df.dropna(subset=["eval_loss"])
        if len(eval_loss) > 0:
            # eval usually logged per epoch; use epoch if available else step
            xs = eval_loss["epoch"] if "epoch" in eval_loss.columns else eval_loss["step"]
            plt.plot(xs, eval_loss["eval_loss"], marker="o", label="eval_loss")

    plt.title(f"Loss Curve - {model_tag}")
    plt.xlabel("Step / Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()


def plot_eval_f1_curves(history_base, history_kg, save_path):
    plt.figure(figsize=(8, 5))

    for hdf, tag in [(history_base, "FaBERT-Base"), (history_kg, "FaBERT-KG")]:
        if "eval_macro_f1" in hdf.columns:
            tmp = hdf.dropna(subset=["eval_macro_f1"])
            if len(tmp) > 0:
                xs = tmp["epoch"] if "epoch" in tmp.columns else tmp["step"]
                plt.plot(xs, tmp["eval_macro_f1"], marker="o", label=tag)

    plt.title("Validation Macro-F1 over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1")
    plt.ylim(0.0, 1.0)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()


# ============================================================
# 2) Train FaBERT-Base
# ============================================================
base_out = train_one_model(
    model_name_or_path=BASE_MODEL_NAME,
    model_tag="FaBERT-Base",
    output_dir=OUTPUT_BASE,
)

# Free before second training
del base_out["trainer"]
free_memory()


# ============================================================
# 3) Train FaBERT-KG
# ============================================================
kg_out = train_one_model(
    model_name_or_path=KG_MODEL_PATH,
    model_tag="FaBERT-KG",
    output_dir=OUTPUT_KG,
)

del kg_out["trainer"]
free_memory()


# ============================================================
# 4) Full comparison tables
# ============================================================
print("\n" + "=" * 70)
print("FINAL TEST COMPARISON")
print("=" * 70)

compare_test = pd.DataFrame({
    "FaBERT-Base": base_out["test_metrics"],
    "FaBERT-KG": kg_out["test_metrics"],
}).T

compare_val = pd.DataFrame({
    "FaBERT-Base": base_out["val_metrics"],
    "FaBERT-KG": kg_out["val_metrics"],
}).T

compare_test["model"] = compare_test.index
compare_val["model"] = compare_val.index

print("\nValidation metrics:")
print(compare_val[["accuracy", "macro_f1", "weighted_f1"]].round(4))

print("\nTest metrics:")
print(compare_test[["accuracy", "macro_f1", "weighted_f1"]].round(4))

# Delta (KG - Base) on test
delta = {
    "accuracy": kg_out["test_metrics"]["accuracy"] - base_out["test_metrics"]["accuracy"],
    "macro_f1": kg_out["test_metrics"]["macro_f1"] - base_out["test_metrics"]["macro_f1"],
    "weighted_f1": kg_out["test_metrics"]["weighted_f1"] - base_out["test_metrics"]["weighted_f1"],
}
delta_df = pd.DataFrame([delta], index=["KG - Base"]).round(4)
print("\nTest delta (KG - Base):")
print(delta_df)

# Winner by primary metric
if kg_out["test_metrics"]["macro_f1"] > base_out["test_metrics"]["macro_f1"]:
    winner = "FaBERT-KG"
elif kg_out["test_metrics"]["macro_f1"] < base_out["test_metrics"]["macro_f1"]:
    winner = "FaBERT-Base"
else:
    winner = "Tie"

print(f"\nPrimary metric: macro_f1")
print(f"Winner on TEST: {winner}")

print("\n" + "-" * 70)
print("Test classification report: FaBERT-Base")
print(base_out["test_report"])
print("-" * 70)
print("Test classification report: FaBERT-KG")
print(kg_out["test_report"])

# Save comparison
compare_test_path = "/kaggle/working/results/compare_test_metrics.csv"
compare_val_path = "/kaggle/working/results/compare_val_metrics.csv"
delta_path = "/kaggle/working/results/compare_test_delta.csv"

compare_test.to_csv(compare_test_path, index=True)
compare_val.to_csv(compare_val_path, index=True)
delta_df.to_csv(delta_path, index=True)

summary = {
    "winner_by_test_macro_f1": winner,
    "base_test": base_out["test_metrics"],
    "kg_test": kg_out["test_metrics"],
    "delta_kg_minus_base_test": delta,
    "settings": {
        "epochs": 2,
        "lr": 2e-5,
        "train_bs": 16,
        "eval_bs": 32,
        "max_length": 128,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "seed": SEED,
        "metric_for_best_model": "macro_f1",
        "split": "70/15/15 stratified",
    },
}
with open("/kaggle/working/results/final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)


# ============================================================
# 5) Plots
# ============================================================
sns.set_style("whitegrid")

# Confusion matrices
plot_confusion(
    base_out["test_cm"],
    "Test Confusion Matrix - FaBERT-Base",
    "/kaggle/working/figures/cm_test_base.png",
)
plot_confusion(
    kg_out["test_cm"],
    "Test Confusion Matrix - FaBERT-KG",
    "/kaggle/working/figures/cm_test_kg.png",
)

# Metric bars
plot_metric_bars(
    compare_test.set_index("model") if "model" in compare_test.columns else compare_test,
    "/kaggle/working/figures/test_metrics_comparison.png",
)

# Loss curves
plot_loss_curves(
    base_out["history_df"],
    "FaBERT-Base",
    "/kaggle/working/figures/loss_base.png",
)
plot_loss_curves(
    kg_out["history_df"],
    "FaBERT-KG",
    "/kaggle/working/figures/loss_kg.png",
)

# Val macro-F1 curves
plot_eval_f1_curves(
    base_out["history_df"],
    kg_out["history_df"],
    "/kaggle/working/figures/val_macro_f1_curves.png",
)


# ============================================================
# 6) Done
# ============================================================
print("\n" + "=" * 70)
print("ALL DONE")
print("=" * 70)
print("Saved models:")
print(" -", OUTPUT_BASE)
print(" -", OUTPUT_KG)
print("\nSaved results:")
print(" - /kaggle/working/results/compare_test_metrics.csv")
print(" - /kaggle/working/results/compare_val_metrics.csv")
print(" - /kaggle/working/results/compare_test_delta.csv")
print(" - /kaggle/working/results/final_summary.json")
print(" - /kaggle/working/results/*_history.csv")
print(" - /kaggle/working/results/*_test_predictions.csv")
print("\nSaved figures:")
print(" - /kaggle/working/figures/cm_test_base.png")
print(" - /kaggle/working/figures/cm_test_kg.png")
print(" - /kaggle/working/figures/test_metrics_comparison.png")
print(" - /kaggle/working/figures/loss_base.png")
print(" - /kaggle/working/figures/loss_kg.png")
print(" - /kaggle/working/figures/val_macro_f1_curves.png")
print(f"\nWinner by TEST macro_f1: {winner}")
